# Feature Engineering EDA

Exploratory analysis of the feature engineering pipeline outputs.

**Prerequisites:** Run `Cleaning_Final.ipynb` then `FeatureENgineering.ipynb` first.

**Sections:**
1. Pre-Analysis Data Checks (nulls, completeness, normality, persona scores)
2. Table Inventory
3. Activity Ontology Tag Distributions
4. Journey State Distributions (30 / 90 / 180-day windows)
5. Developer Personas
6. Dormancy & Activation Analysis
7. State Transition Patterns
8. Key Feature Distributions
9. Score Axis Decomposition
10. Lane x Journey Stage Cross-Analysis
11. Activity Breadth and Cadence
12. Marketing Touch vs Self-Directed Activity


## Section 0 — Setup

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

con = duckdb.connect("developer_project.duckdb")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

EXPECTED_TABLES = [
    "activity_enriched_v1", "activity_ontology_v1", "developer_universe_v1",
    "dev_features_30d_v1",  "dev_features_90d_v1",  "dev_features_180d_v1",
    "dev_profile_30d_v1",   "dev_profile_90d_v1",   "dev_profile_180d_v1",
    "dev_features_lifetime_v1", "dev_persona_v1",
    "dev_transition_v1", "dev_profile_final_v2",
    "dev_period_30d_features_v1", "dev_period_30d_profile_v1",
    "dev_period_30d_transitions_v1", "dev_weekly_features_v1",
    "dev_meaningful_week_v1", "dev_activation_v1",
    "dev_dormancy_base_v1", "dev_dormancy_status_v1",
    "dev_profile_final_v3",
]

available = set(con.execute("SHOW TABLES").fetchdf().iloc[:, 0])
missing   = [t for t in EXPECTED_TABLES if t not in available]
present   = [t for t in EXPECTED_TABLES if t in available]

print(f"Tables present : {len(present)} / {len(EXPECTED_TABLES)}")
if missing:
    print(f"Tables missing : {missing}")
else:
    print("All expected tables found.")

---
## Section 1 — Pre-Analysis Data Checks

Before the feature-level analysis, validate the quality of `dev_features_lifetime_v1`:
null rates, row completeness, distribution shape (normality), and persona score structure.


In [ ]:
if "dev_features_lifetime_v1" in present:
    KEY_COLS = [
        "lifetime_activity_count", "lifetime_activity_score_sum",
        "lifetime_learn_count", "lifetime_evaluate_count",
        "lifetime_build_count", "lifetime_champion_count",
        "lifetime_high_effort_count",
        "cuda_score", "genai_score", "robotics_score",
        "simulation_score", "learning_community_score",
    ]

    # Only check columns that actually exist
    all_cols = [r[0] for r in con.execute("DESCRIBE dev_features_lifetime_v1").fetchall()]
    check_cols = [c for c in KEY_COLS if c in all_cols]

    null_rows = []
    total_n = con.execute("SELECT COUNT(*) FROM dev_features_lifetime_v1").fetchone()[0]
    for col in check_cols:
        n_null = con.execute(f"SELECT COUNT(*) FROM dev_features_lifetime_v1 WHERE {col} IS NULL").fetchone()[0]
        null_rows.append({"column": col, "null_count": n_null,
                          "null_pct": round(n_null / total_n * 100, 2)})

    null_df = pd.DataFrame(null_rows)
    print(f"Total rows: {total_n:,}")
    display(null_df.sort_values("null_pct", ascending=False))
else:
    print("dev_features_lifetime_v1 not available.")


In [ ]:
if "dev_features_lifetime_v1" in present:
    # How many rows survive if we require the core activity columns to be non-null and > 0
    filters = {
        "Any activity recorded (activity_count > 0)":
            "lifetime_activity_count > 0",
        "Has at least one scored activity (score_sum > 0)":
            "lifetime_activity_count > 0 AND lifetime_activity_score_sum > 0",
        "Has journey-signal activity (learn+build+eval+champion > 0)":
            "lifetime_activity_count > 0 AND "
            "(lifetime_learn_count + lifetime_evaluate_count + "
            " lifetime_build_count + lifetime_champion_count) > 0",
        "Has any persona lane score > 0":
            "lifetime_activity_count > 0 AND "
            "(cuda_score + genai_score + robotics_score + "
            " simulation_score + learning_community_score) > 0",
    }

    total_n = con.execute("SELECT COUNT(*) FROM dev_features_lifetime_v1").fetchone()[0]
    print(f"{'Filter':<60} {'Rows kept':>12} {'% kept':>8} {'% dropped':>10}")
    print("-" * 95)
    for label, where in filters.items():
        try:
            n = con.execute(f"SELECT COUNT(*) FROM dev_features_lifetime_v1 WHERE {where}").fetchone()[0]
            print(f"{label:<60} {n:>12,} {n/total_n*100:>7.1f}% {(total_n-n)/total_n*100:>9.1f}%")
        except Exception as e:
            print(f"{label:<60} ERROR: {e}")
else:
    print("dev_features_lifetime_v1 not available.")


In [ ]:
if "dev_features_lifetime_v1" in present:
    DIST_COLS = [
        "lifetime_activity_count", "lifetime_activity_score_sum",
        "lifetime_learn_count", "lifetime_build_count",
        "lifetime_champion_count", "lifetime_high_effort_count",
    ]
    all_cols = [r[0] for r in con.execute("DESCRIBE dev_features_lifetime_v1").fetchall()]
    dist_cols = [c for c in DIST_COLS if c in all_cols]

    df = con.execute(f"""
        SELECT {', '.join(dist_cols)}
        FROM dev_features_lifetime_v1
        WHERE lifetime_activity_count > 0
        USING SAMPLE 30000
    """).df()

    import scipy.stats as stats

    n_cols = len(dist_cols)
    fig, axes = plt.subplots(2, n_cols, figsize=(4 * n_cols, 8))
    fig.suptitle("Feature Distributions — Raw vs Log-transformed (active devs, 30k sample)", fontsize=12)

    skew_rows = []
    for j, col in enumerate(dist_cols):
        raw = df[col].dropna()
        log = np.log1p(raw)

        # Raw histogram
        axes[0, j].hist(raw.clip(upper=raw.quantile(0.99)), bins=40,
                        color="#90CAF9", edgecolor="white")
        axes[0, j].set_title(col.replace("lifetime_",""), fontsize=8)
        if j == 0:
            axes[0, j].set_ylabel("Count (raw)")

        # Log1p histogram
        axes[1, j].hist(log, bins=40, color="#A5D6A7", edgecolor="white")
        if j == 0:
            axes[1, j].set_ylabel("Count (log1p)")

        skew_rows.append({
            "column": col.replace("lifetime_",""),
            "raw_skew":   round(float(stats.skew(raw)), 2),
            "log_skew":   round(float(stats.skew(log)), 2),
            "median":     round(float(raw.median()), 1),
            "p99":        round(float(raw.quantile(0.99)), 1),
        })

    plt.tight_layout()
    plt.show()

    skew_df = pd.DataFrame(skew_rows)
    print("\nSkewness summary (|skew| < 1 = roughly normal, > 2 = heavily skewed):")
    display(skew_df)
else:
    print("dev_features_lifetime_v1 not available.")


In [ ]:
if "dev_features_lifetime_v1" in present:
    LANE_COLS = ["cuda_score", "genai_score", "robotics_score",
                 "simulation_score", "learning_community_score"]
    all_cols = [r[0] for r in con.execute("DESCRIBE dev_features_lifetime_v1").fetchall()]
    lane_cols = [c for c in LANE_COLS if c in all_cols]

    df = con.execute(f"""
        SELECT {', '.join(lane_cols)}
        FROM dev_features_lifetime_v1
        WHERE lifetime_activity_count > 0
        USING SAMPLE 50000
    """).df()

    # Count how many lanes each developer has a non-zero score in
    df["lanes_active"] = (df[lane_cols] > 0).sum(axis=1)

    # Top persona score as a fraction of total persona score
    df["total_lane_score"] = df[lane_cols].sum(axis=1)
    df["top_lane_score"]   = df[lane_cols].max(axis=1)
    df["top_lane_pct"]     = df["top_lane_score"] / df["total_lane_score"].replace(0, float("nan")) * 100

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Persona Score Structure (active devs, 50k sample)", fontsize=13)

    # 1. How many lanes active per developer
    lane_dist = df["lanes_active"].value_counts().sort_index()
    axes[0].bar(lane_dist.index.astype(str), lane_dist.values / len(df) * 100,
                color="#42A5F5", edgecolor="white")
    axes[0].set_xlabel("Number of lanes with score > 0")
    axes[0].set_ylabel("% of active developers")
    axes[0].set_title("Lanes Active per Developer")

    # 2. Top lane dominance (% of total persona score)
    axes[1].hist(df["top_lane_pct"].dropna(), bins=20, color="#AB47BC", edgecolor="white")
    axes[1].axvline(80, color="red", linestyle="--", label="80% threshold")
    axes[1].set_xlabel("Top lane % of total persona score")
    axes[1].set_ylabel("Developers")
    axes[1].set_title("Top Lane Dominance")
    axes[1].legend()

    # 3. Score distribution per lane (box plots)
    log_scores = np.log1p(df[lane_cols])
    log_scores.columns = [c.replace("_score","").replace("learning_community","comm") for c in lane_cols]
    log_scores.boxplot(ax=axes[2], vert=True)
    axes[2].set_ylabel("log1p(persona score)")
    axes[2].set_title("Per-Lane Score Distribution")
    axes[2].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.show()

    single_lane_pct = (df["lanes_active"] == 1).sum() / len(df) * 100
    multi_lane_pct  = (df["lanes_active"] >  1).sum() / len(df) * 100
    zero_lane_pct   = (df["lanes_active"] == 0).sum() / len(df) * 100
    dominant_pct    = (df["top_lane_pct"] >= 80).sum() / df["top_lane_pct"].notna().sum() * 100

    print(f"Developers with 0 active lanes:  {zero_lane_pct:.1f}%")
    print(f"Developers with 1 active lane:   {single_lane_pct:.1f}%")
    print(f"Developers with 2+ active lanes: {multi_lane_pct:.1f}%")
    print(f"Developers with dominant lane (>=80% of score): {dominant_pct:.1f}%")
else:
    print("dev_features_lifetime_v1 not available.")


---
## Section 2 — Table Inventory

In [ ]:
# Row counts for every feature engineering output table
rows = []
for t in present:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    rows.append({"table": t, "row_count": n})

inventory = pd.DataFrame(rows).set_index("table")
print("Feature table row counts:")
display(inventory)

In [ ]:
# Developer universe coverage
universe = con.execute("SELECT COUNT(*) FROM developer_universe_v1").fetchone()[0]
contact  = con.execute("SELECT COUNT(*) FROM contact_final").fetchone()[0]
print(f"Developer universe (union of contact + activity): {universe:,}")
print(f"contact_final:                                    {contact:,}")
print(f"Coverage:                                         {universe/contact*100:.1f}%")

---
## Section 3 — Activity Ontology Tags

`activity_ontology_v1` assigns four behavioral tags to every activity event.
These tags drive journey state assignment and persona scoring.

In [ ]:
# Distribution of each ontology tag
tags = ["journey_signal", "effort_level", "persona_hint", "modality"]

for tag in tags:
    df = con.execute(f"""
        SELECT {tag}, COUNT(*) AS rows,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM activity_ontology_v1
        GROUP BY {tag}
        ORDER BY rows DESC
    """).df()
    print(f"\n{tag}")
    display(df)

In [ ]:
# Bar charts for all four tags
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, tag in zip(axes.flatten(), tags):
    df = con.execute(f"""
        SELECT {tag} AS label, COUNT(*) AS rows
        FROM activity_ontology_v1
        GROUP BY {tag} ORDER BY rows DESC
    """).df()
    ax.barh(df["label"].astype(str), df["rows"])
    ax.set_title(tag)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
    ax.invert_yaxis()

plt.suptitle("Activity Ontology Tag Distributions", fontsize=14)
plt.tight_layout()
plt.show()

---
## Section 4 — Journey State Distributions

Journey states are assigned per developer for three cumulative windows (30 / 90 / 180 days).
States: **Champion, Build, Evaluate, Learn, Passive, Dormant**

In [ ]:
# Journey state breakdown for each window
for d in [30, 90, 180]:
    table = f"dev_profile_{d}d_v1"
    if table not in present:
        print(f"{table} not found, skipping")
        continue
    df = con.execute(f"""
        SELECT journey_state,
               COUNT(*) AS developers,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM {table}
        GROUP BY journey_state
        ORDER BY developers DESC
    """).df()
    print(f"\n{d}-day window")
    display(df)

In [ ]:
# Side-by-side comparison of journey states across windows
all_states = set()
window_data = {}
for d in [30, 90, 180]:
    table = f"dev_profile_{d}d_v1"
    if table not in present:
        continue
    df = con.execute(f"""
        SELECT journey_state, COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS pct
        FROM {table} GROUP BY journey_state
    """).df().set_index("journey_state")["pct"]
    window_data[f"{d}d"] = df
    all_states.update(df.index)

if window_data:
    comp = pd.DataFrame(window_data, index=sorted(all_states)).fillna(0)
    comp.plot(kind="bar", figsize=(12, 5))
    plt.title("Journey State % by Window")
    plt.ylabel("% of developers")
    plt.xticks(rotation=30)
    plt.legend(title="Window")
    plt.tight_layout()
    plt.show()

---
## Section 5 — Developer Personas

Personas are assigned from lifetime activity using weighted keyword scoring across six lanes:
**CUDA, GenAI, Robotics, Simulation, Learning/Community, Other**

In [ ]:
if "dev_persona_v1" in present:
    df = con.execute("""
        SELECT persona,
               COUNT(*) AS developers,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM dev_persona_v1
        GROUP BY persona
        ORDER BY developers DESC
    """).df()
    print("Persona distribution:")
    display(df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(df["persona"], df["developers"])
    axes[0].set_title("Developer Count by Persona")
    axes[0].tick_params(axis="x", rotation=30)
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

    axes[1].pie(df["developers"], labels=df["persona"], autopct="%1.1f%%", startangle=140)
    axes[1].set_title("Persona Share")

    plt.tight_layout()
    plt.show()

In [ ]:
if "dev_persona_v1" in present:
    # Confidence tier breakdown
    cols = con.execute("DESCRIBE dev_persona_v1").df()["column_name"].tolist()

    if "persona_confidence_tier" in cols:
        conf = con.execute("""
            SELECT persona_confidence_tier,
                   COUNT(*) AS developers,
                   ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
            FROM dev_persona_v1
            GROUP BY persona_confidence_tier
            ORDER BY developers DESC
        """).df()
        print("Persona confidence tiers:")
        display(conf)

    if "mixed_persona_flag" in cols:
        mixed = con.execute("""
            SELECT mixed_persona_flag, COUNT(*) AS developers
            FROM dev_persona_v1 GROUP BY mixed_persona_flag
        """).df()
        print("\nMixed persona flag:")
        display(mixed)

---
## Section 6 — Dormancy & Activation Analysis

Dormancy uses a survival-based framework with two thresholds:
- **Active**: last meaningful week < 56 days ago
- **At-Risk**: 56–83 days
- **Dormant**: ≥ 84 days
- **Unactivated**: never passed the activation gate

In [ ]:
if "dev_activation_v1" in present:
    act = con.execute("""
        SELECT is_activated,
               COUNT(*) AS developers,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM dev_activation_v1
        GROUP BY is_activated ORDER BY is_activated
    """).df()
    print("Activation status:")
    display(act)

    if "activation_reason" in con.execute("DESCRIBE dev_activation_v1").df()["column_name"].tolist():
        reason = con.execute("""
            SELECT activation_reason, COUNT(*) AS developers
            FROM dev_activation_v1
            GROUP BY activation_reason ORDER BY developers DESC
        """).df()
        print("\nActivation reason breakdown:")
        display(reason)

In [ ]:
if "dev_dormancy_status_v1" in present:
    dorm = con.execute("""
        SELECT dormancy_status,
               COUNT(*) AS developers,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM dev_dormancy_status_v1
        GROUP BY dormancy_status ORDER BY developers DESC
    """).df()
    print("Dormancy status breakdown:")
    display(dorm)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(dorm["dormancy_status"], dorm["developers"])
    axes[0].set_title("Developers by Dormancy Status")
    axes[0].tick_params(axis="x", rotation=20)
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

    axes[1].pie(dorm["developers"], labels=dorm["dormancy_status"], autopct="%1.1f%%", startangle=140)
    axes[1].set_title("Dormancy Share")

    plt.tight_layout()
    plt.show()

In [ ]:
# Distribution of days_since_last_meaningful_week for activated developers
if "dev_dormancy_base_v1" in present:
    days_df = con.execute("""
        SELECT days_since_last_meaningful_week
        FROM dev_dormancy_base_v1
        WHERE days_since_last_meaningful_week IS NOT NULL
          AND days_since_last_meaningful_week <= 365
    """).df()

    plt.figure(figsize=(12, 4))
    plt.hist(days_df["days_since_last_meaningful_week"], bins=60, edgecolor="none")
    plt.axvline(56, color="orange", linestyle="--", label="At-risk threshold (56d)")
    plt.axvline(84, color="red",    linestyle="--", label="Dormant threshold (84d)")
    plt.xlabel("Days since last meaningful active week")
    plt.ylabel("Developers")
    plt.title("Days Since Last Meaningful Activity (activated developers, ≤365d shown)")
    plt.legend()
    plt.tight_layout()
    plt.show()

---
## Section 7 — State Transition Patterns

In [ ]:
# Cumulative window transitions: 30d → 90d → 180d
if "dev_transition_v1" in present:
    cols = con.execute("DESCRIBE dev_transition_v1").df()["column_name"].tolist()
    print("dev_transition_v1 columns:", cols)

    if "state_30d" in cols and "state_90d" in cols:
        t30_90 = con.execute("""
            SELECT state_30d, state_90d, COUNT(*) AS developers
            FROM dev_transition_v1
            GROUP BY state_30d, state_90d
            ORDER BY developers DESC
            LIMIT 20
        """).df()
        print("\nTop transitions: 30d → 90d")
        display(t30_90)

    if "state_90d" in cols and "state_180d" in cols:
        t90_180 = con.execute("""
            SELECT state_90d, state_180d, COUNT(*) AS developers
            FROM dev_transition_v1
            GROUP BY state_90d, state_180d
            ORDER BY developers DESC
            LIMIT 20
        """).df()
        print("\nTop transitions: 90d → 180d")
        display(t90_180)

In [ ]:
# Period-to-period (non-cumulative 30-day buckets) transitions
if "dev_period_30d_transitions_v1" in present:
    period_t = con.execute("""
        SELECT from_state, to_state, COUNT(*) AS transitions,
               ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM dev_period_30d_transitions_v1
        GROUP BY from_state, to_state
        ORDER BY transitions DESC
        LIMIT 25
    """).df()
    print("Top period-to-period transitions:")
    display(period_t)

---
## Section 8 — Key Feature Distributions

In [ ]:
# Lifetime feature summary statistics
if "dev_features_lifetime_v1" in present:
    summary = con.execute("""
        SELECT
            COUNT(*) AS developers,
            MIN(lifetime_activity_count)     AS min_activities,
            MAX(lifetime_activity_count)     AS max_activities,
            AVG(lifetime_activity_count)     AS avg_activities,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY lifetime_activity_count) AS median_activities,
            MIN(lifetime_activity_score_sum) AS min_score,
            MAX(lifetime_activity_score_sum) AS max_score,
            AVG(lifetime_activity_score_sum) AS avg_score
        FROM dev_features_lifetime_v1
    """).df()
    print("Lifetime feature summary:")
    display(summary)

In [ ]:
# Average activity count per developer across time windows
window_stats = []
for d in [30, 90, 180]:
    table = f"dev_features_{d}d_v1"
    if table not in present:
        continue
    cols = con.execute(f"DESCRIBE {table}").df()["column_name"].tolist()
    if "activity_count_total" not in cols:
        continue
    row = con.execute(f"""
        SELECT
            '{d}d' AS window,
            COUNT(*) AS developers,
            AVG(activity_count_total) AS avg_activity_count,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY activity_count_total) AS median_activity_count,
            SUM(CASE WHEN activity_count_total = 0 THEN 1 ELSE 0 END) AS zero_activity_developers
        FROM {table}
    """).fetchone()
    window_stats.append(row)

if window_stats:
    print("Activity count stats by window:")
    display(pd.DataFrame(window_stats, columns=["window", "developers", "avg_activity_count",
                                                 "median_activity_count", "zero_activity_developers"]))

In [ ]:
# Final profile summary
final_table = "dev_profile_final_v3" if "dev_profile_final_v3" in present else \
              "dev_profile_final_v2" if "dev_profile_final_v2" in present else None

if final_table:
    print(f"Final profile table: {final_table}")
    n = con.execute(f"SELECT COUNT(*) FROM {final_table}").fetchone()[0]
    print(f"Rows: {n:,}")
    print("\nColumns:")
    display(con.execute(f"DESCRIBE {final_table}").df()[["column_name", "column_type"]])
    print("\nSample rows:")
    display(con.execute(f"SELECT * FROM {final_table} LIMIT 5").df())

---
## Section 9 — Score Axis Decomposition

The PDF framework replaces a single `activity_score` with four independent axes so developers
with the same lifetime score can still be at very different journey stages:

| Axis | Signal |
|------|--------|
| Learn | `journey_signal` in Learn / Evaluate |
| Build | `journey_signal` = Build |
| Community | `journey_signal` = Champion |
| High-effort | `effort_level` = High |


In [ ]:
if "dev_features_lifetime_v1" in present:
    df = con.execute("""
        SELECT lifetime_learn_count, lifetime_evaluate_count,
               lifetime_build_count, lifetime_champion_count,
               lifetime_high_effort_count
        FROM dev_features_lifetime_v1
        WHERE lifetime_activity_count > 0
        USING SAMPLE 50000
    """).df()

    total_sig = (
        df["lifetime_learn_count"] + df["lifetime_evaluate_count"] +
        df["lifetime_build_count"] + df["lifetime_champion_count"]
    ).replace(0, float("nan"))
    df["learn_pct"]     = (df["lifetime_learn_count"] + df["lifetime_evaluate_count"]) / total_sig * 100
    df["build_pct"]     = df["lifetime_build_count"]   / total_sig * 100
    df["community_pct"] = df["lifetime_champion_count"] / total_sig * 100

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Score Axis Decomposition (active devs, 50k sample)", fontsize=13)

    means = {"Learn+Eval": df["learn_pct"].mean(),
             "Build": df["build_pct"].mean(),
             "Community": df["community_pct"].mean()}
    axes[0].bar(means.keys(), means.values(), color=["#42A5F5", "#66BB6A", "#EF5350"])
    axes[0].set_ylabel("Mean % of signalled activities")
    axes[0].set_title("Average Mix Across Active Developers")

    sample = df.dropna(subset=["learn_pct","build_pct","community_pct"]).sample(min(20000, len(df)))
    sc = axes[1].scatter(sample["learn_pct"], sample["build_pct"],
                         c=sample["community_pct"], cmap="RdYlGn", alpha=0.3, s=3)
    plt.colorbar(sc, ax=axes[1], label="Community %")
    axes[1].set_xlabel("Learn + Evaluate % of activities")
    axes[1].set_ylabel("Build % of activities")
    axes[1].set_title("Learn vs Build (coloured by Community)")

    hec = df["lifetime_high_effort_count"].clip(upper=50)
    axes[2].hist(hec, bins=30, color="#FFA726", edgecolor="white")
    axes[2].set_xlabel("High-effort activity count (capped at 50)")
    axes[2].set_ylabel("Developers")
    axes[2].set_title("High-effort Activity Distribution")

    plt.tight_layout()
    plt.show()
else:
    print("dev_features_lifetime_v1 not available.")


---
## Section 10 — Lane x Journey Stage Cross-Analysis

Combining persona lane with journey state shows where each technical community sits in its
adoption cycle and which lanes have the most developers stuck in early stages vs already
building or championing.


In [ ]:
final_t = ("dev_profile_final_v3" if "dev_profile_final_v3" in present else
           "dev_profile_final_v2" if "dev_profile_final_v2" in present else None)

if "dev_persona_v1" in present and final_t:
    cross = con.execute(f"""
        SELECT p.persona,
               f.journey_state_90d AS journey_state,
               COUNT(*) AS n
        FROM dev_persona_v1 p
        JOIN {final_t} f USING (developer_id)
        WHERE p.persona IS NOT NULL AND f.journey_state_90d IS NOT NULL
        GROUP BY p.persona, f.journey_state_90d
    """).df()

    pivot = cross.pivot_table(index="persona", columns="journey_state", values="n", fill_value=0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    order = [c for c in ["Champion","Build","Evaluate","Learn","Passive","Dormant"] if c in pivot_pct.columns]
    if order:
        pivot_pct = pivot_pct[order]
    colors = ["#AB47BC","#EF5350","#FFEE58","#42A5F5","#BDBDBD","#78909C"]

    ax = pivot_pct.plot(kind="bar", stacked=True, figsize=(12, 5),
                        color=colors[:len(pivot_pct.columns)], edgecolor="white", linewidth=0.4)
    ax.set_xlabel("Developer persona (technical lane)")
    ax.set_ylabel("% of developers")
    ax.set_title("Journey State by Lane (90-day window)")
    ax.legend(title="Journey state", bbox_to_anchor=(1.02,1), loc="upper left")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("dev_persona_v1 or final profile table not available.")


---
## Section 11 — Activity Breadth and Cadence

**Breadth** measures cross-lane diversity. A developer touching CUDA, GenAI, and Simulation
is likely an enterprise platform team or broad researcher.

**Cadence** measures engagement regularity via the coefficient of variation (CV) of weekly
activity counts. Low CV = regular weekly engagement. High CV = occasional bursts.


In [ ]:
if "dev_features_lifetime_v1" in present:
    breadth = con.execute("""
        SELECT
            (CASE WHEN cuda_score > 0 THEN 1 ELSE 0 END
           + CASE WHEN genai_score > 0 THEN 1 ELSE 0 END
           + CASE WHEN robotics_score > 0 THEN 1 ELSE 0 END
           + CASE WHEN simulation_score > 0 THEN 1 ELSE 0 END
           + CASE WHEN learning_community_score > 0 THEN 1 ELSE 0 END) AS active_lanes,
            COUNT(*) AS n
        FROM dev_features_lifetime_v1
        WHERE lifetime_activity_count > 0
        GROUP BY active_lanes ORDER BY active_lanes
    """).df()
    total_b = breadth["n"].sum()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Activity Breadth and Cadence", fontsize=13)

    axes[0].bar(breadth["active_lanes"].astype(str), breadth["n"] / total_b * 100,
               color="#42A5F5", edgecolor="white")
    axes[0].set_xlabel("Number of distinct technical lanes active")
    axes[0].set_ylabel("% of active developers")
    axes[0].set_title("Lane Breadth Distribution")

    if "dev_weekly_features_v1" in present:
        cadence = con.execute("""
            SELECT cv_bucket, COUNT(*) AS n FROM (
                SELECT developer_id,
                       ROUND(STDDEV(activity_count_total) / NULLIF(AVG(activity_count_total),0), 1) AS cv_bucket
                FROM dev_weekly_features_v1
                GROUP BY developer_id
                HAVING COUNT(*) >= 4
            ) WHERE cv_bucket IS NOT NULL AND cv_bucket <= 5
            GROUP BY cv_bucket ORDER BY cv_bucket
        """).df()
        if len(cadence):
            axes[1].bar(cadence["cv_bucket"].astype(str), cadence["n"],
                       color="#FFA726", edgecolor="white")
            axes[1].set_xlabel("Weekly activity CV (0=regular, >2=bursty)")
            axes[1].set_ylabel("Developers (>= 4 active weeks)")
            axes[1].set_title("Activity Cadence Distribution")
        else:
            axes[1].text(0.5, 0.5, "No data", ha="center", va="center")
            axes[1].axis("off")
    else:
        axes[1].text(0.5, 0.5, "dev_weekly_features_v1 not available", ha="center", va="center")
        axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    multi = breadth[breadth["active_lanes"] > 1]["n"].sum() / total_b * 100
    print(f"Developers active in > 1 lane: {multi:.1f}%")
else:
    print("dev_features_lifetime_v1 not available.")


---
## Section 12 — Marketing Touch vs Self-Directed Activity

Some developers appear highly engaged only because they respond to campaigns. Using
`lead_source` from `activity_ontology_v1` to separate campaign-driven from organic activity.


In [ ]:
ao_cols = {r[0] for r in con.execute("DESCRIBE activity_ontology_v1").fetchall()}
if "lead_source" in ao_cols:
    intent = con.execute("""
        SELECT developer_id,
               COUNT(*) AS total_act,
               SUM(CASE WHEN lead_source NOT IN ('unknown','') AND lead_source IS NOT NULL
                        THEN 1 ELSE 0 END) AS campaign_act
        FROM activity_ontology_v1
        WHERE developer_id IS NOT NULL
        GROUP BY developer_id
    """).df()
    intent["campaign_pct"] = intent["campaign_act"] / intent["total_act"].clip(lower=1) * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Marketing Touch vs Self-Directed Activity", fontsize=13)

    axes[0].hist(intent["campaign_pct"].clip(upper=100), bins=20,
                 color="#EF5350", edgecolor="white")
    axes[0].set_xlabel("Campaign-driven activity % per developer")
    axes[0].set_ylabel("Developers")
    axes[0].set_title("Campaign Activity Distribution")

    cp = intent["campaign_pct"]
    buckets = [("Fully organic (0%)",    (cp == 0).sum()),
               ("Mostly organic (<25%)", ((cp > 0) & (cp < 25)).sum()),
               ("Mixed (25-75%)",        ((cp >= 25) & (cp <= 75)).sum()),
               ("Mostly campaign (>75%)",(cp > 75).sum())]
    lbs, cnts = zip(*buckets)
    axes[1].barh(lbs, cnts, color=["#66BB6A","#42A5F5","#FFA726","#EF5350"])
    axes[1].set_xlabel("Number of developers")
    axes[1].set_title("Intent Segmentation by Campaign Mix")

    plt.tight_layout()
    plt.show()
    print(f"Fully organic: {(cp==0).sum()/len(intent)*100:.1f}%")
else:
    print("lead_source column not in activity_ontology_v1 — skipping.")


In [ ]:
con.close()
print("Done.")